<a href="https://colab.research.google.com/github/ebuseyy/SGuA-projects/blob/main/BeforeRevision/BOA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np

# Define the number of relays and the number of butterflies
num_relays = 6
num_butterflies = 100

# Define the maximum and minimum values for parameters
ti_min = 1.0
ti_max = 2.2
td_min = 0.2
td_max = 1.0
Ikd1 = 1263
Ikd2_6 = 5639
Ip_min = [117.152, 523, 400, 500, 600, 400]
Ip_max = [128.65, 575, 420, 510, 610, 420]

def calculate_ti(td, Ikd, RIP):
    ti = [0] * num_relays
    for k in range(num_relays):
        x = (Ikd[k] / RIP[k]) ** 0.02
        calculated_ti = 0.14 * td[k] / (x - 1)
        # Soft constraint handling
        if calculated_ti < ti_min:
            ti[k] = ti_min + (ti_min - calculated_ti) * 0.1  # Penalize but don't set to boundary
        elif calculated_ti > ti_max:
            ti[k] = ti_max - (calculated_ti - ti_max) * 0.1  # Penalize but don't set to boundary
        else:
            ti[k] = calculated_ti
    return ti

def evaluate_solution(td, Ikd, RIP):
    # Ensure td is within the specified range (Constraint 2)
    if any(t < td_min or t > td_max for t in td):
        return float('inf')  # Penalize invalid solutions

    # Ensure Ikd is within the specified range
    if any(Ikd_val < Ikd2_6 for Ikd_val in Ikd[1:]):
        return float('inf')  # Penalize invalid solutions

    # Ensure Ip is within the specified range (Constraint 3)
    for i in range(num_relays):
        if RIP[i] < Ip_min[i] or RIP[i] > Ip_max[i]:
            return float('inf')  # Penalize invalid solutions

    # Calculate relay operation times
    relay_times = calculate_ti(td, Ikd, RIP)

    # Check if relay_times are within the specified range (Constraint 1)
    for i in range(num_relays):
        if relay_times[i] < ti_min or relay_times[i] > ti_max:
            return float('inf')  # Penalize invalid solutions

    # Apply the second constraint
    # t1 should be greater than t2 by at least 0.3 seconds
    if relay_times[0] <= relay_times[1] + 0.3:
        return float('inf')

    # Apply the updated constraint: t2 should be greater than other relays (t3, t4, t5, t6) by at least 0.3 seconds
    min_other_relays = min(relay_times[2:])  # Find the minimum of t3, t4, t5, t6
    if relay_times[1] <= min_other_relays + 0.3:
        return float('inf')

    # Minimize the sum of relay operation times
    return sum(relay_times)

class Butterfly:
    def __init__(self, td, Ikd, RIP):
        self.td = td
        self.Ikd = Ikd
        self.RIP = RIP
        self.fitness = evaluate_solution(td, Ikd, RIP)

def run_boa_algorithm(num_iterations=1000):
    # Initialize butterflies with random solutions
    butterflies = []
    for _ in range(num_butterflies):
        td = np.random.uniform(low=td_min, high=td_max, size=num_relays)
        Ikd = [Ikd1] + [Ikd2_6] * (num_relays - 1)
        RIP = [np.random.uniform(low=Ip_min[i], high=Ip_max[i]) for i in range(num_relays)]
        butterfly = Butterfly(td, Ikd, RIP)
        butterflies.append(butterfly)

    # Main BOA loop
    for iteration in range(num_iterations):
        for i in range(num_butterflies):
            # Create a new butterfly by perturbing the current one
            new_td = butterflies[i].td + np.random.uniform(-0.1, 0.1, size=num_relays)
            new_td = np.maximum(td_min, np.minimum(new_td, td_max))  # Ensure td values are within bounds
            new_Ikd = butterflies[i].Ikd
            new_RIP = butterflies[i].RIP + np.random.uniform(-1.0, 1.0, size=num_relays)
            new_RIP = np.maximum(Ip_min, np.minimum(new_RIP, Ip_max))  # Ensure RIP values are within bounds
            new_fitness = evaluate_solution(new_td, new_Ikd, new_RIP)

            # Update the butterfly's solution if it's better
            if new_fitness < butterflies[i].fitness:
                butterflies[i] = Butterfly(new_td, new_Ikd, new_RIP)

    # Find the best solution
    best_butterfly = min(butterflies, key=lambda x: x.fitness)

    # Calculate relay operating times using the best solution
    relay_ti = calculate_ti(best_butterfly.td, best_butterfly.Ikd, best_butterfly.RIP)

    return best_butterfly, relay_ti

# Call the BOA algorithm function
best_butterfly, relay_ti = run_boa_algorithm()

# Print the best solution and relay operating times
if best_butterfly is not None:
    print("Table 1. BOA Results.")
    print("{:<10} {:<12} {:<10} {:<10}".format("Relay", "td", "Ip(A)", "Ti(s)"))
    for i in range(num_relays):
        print("{:<10} {:<12.6f} {:<10.3f} {:<10.3f}".format("Relay-{}".format(i + 1), best_butterfly.td[i], best_butterfly.RIP[i], relay_ti[i]))
    print("Best Fitness (minimized relay operation times):", best_butterfly.fitness)


Table 1. BOA Results.
Relay      td           Ip(A)      Ti(s)     
Relay-1    0.553734     122.701    1.624     
Relay-2    0.454417     523.549    1.307     
Relay-3    0.350413     415.485    1.008     
Relay-4    0.329208     505.706    1.007     
Relay-5    0.249783     604.124    1.023     
Relay-6    0.361684     414.048    1.006     
Best Fitness (minimized relay operation times): 6.974919646804102
